# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to programmatically explore, load, and analyze the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset focuses on protein abundance, modifications, and sequences in human samples, as revealed by mass spectrometry after extracellular vesicle isolation.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare to extract records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict!

print(f"Dataset: {metadata.name}\nPublished: {metadata.datePublished}\n\n{metadata.description}")

## 2. Data Overview
Use the Croissant metadata to review available record sets (*tables*), their `@id`s, and the fields (columns) within each.

Below, we list all record sets and their field `@id`s. You can use these IDs to load and refer to data consistently throughout this notebook.

In [ ]:
# List all available record sets and their fields by @id
if not hasattr(metadata, "recordSet") or not metadata.recordSet:
    print("No record sets are defined in the root metadata. Loading using schema traversal...\n")
    # This is a fallback to get recordSets via the internal mlcroissant dataset implementation
    from mlcroissant._src.structure_graph import constants
    graph = dataset._graph
    recordset_nodes = [n for n in graph.nodes if n.type == constants.RECORD_SET_TYPE]
    for rset in recordset_nodes:
        print(f"RecordSet name: {rset.props.get('name', '')}\n  @id: {rset.id}")
        columns = rset.props.get("field", [])
        if isinstance(columns, dict): columns = [columns]
        for f in columns:
            if isinstance(f, dict):
                print(f"    field @id: {f.get('@id', None)}")
            elif isinstance(f, str):
                print(f"    field @id: {f}")
        print()
    # Save all recordset IDs for further use
    recordset_ids = [r.id for r in recordset_nodes]
else:
    recordset_ids = []
    for recordset in metadata.recordSet:
        # recordset may be dict or a croissant object (depends on load)
        rid = getattr(recordset, '@id', None) or recordset.get('@id')
        rname = getattr(recordset, 'name', None) or recordset.get('name')
        print(f"RecordSet name: {rname}\n  @id: {rid}")
        fields = getattr(recordset, 'field', None) or recordset.get('field', [])
        if isinstance(fields, dict): fields = [fields]
        for field in fields:
            fid = getattr(field, '@id', None) or field.get('@id')
            print(f"    field @id: {fid}")
        print()
        if rid:
            recordset_ids.append(rid)

# For this notebook, we'll use the auto-detected record set IDs from above
print(f"RecordSets: {recordset_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

You must refer to record sets by their `@id` as identified in the previous overview.

In [ ]:
# Prepare to extract all record sets into DataFrames
dataframes = {}

for rset_id in recordset_ids:
    try:
        records_iter = dataset.records(record_set=rset_id)
        df = pd.DataFrame(records_iter)
        print(f"Loaded {len(df)} records for record set: {rset_id}")
        dataframes[rset_id] = df
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Could not load record set {rset_id}: {e}\n")

# Pick one record set to demonstrate further analysis on
main_record_set_id = recordset_ids[0] if recordset_ids else None

if main_record_set_id:
    print(f"Sample rows from main record set (@id={main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform EDA to gain insights. Common operations include filtering by field values, normalizing numeric columns, detecting outliers, and grouping. We'll work with the main record set loaded above.

> **Note:** You must reference specific fields or columns by their `@id`, as found in the previous overview.

In [ ]:
df = dataframes[main_record_set_id]

# Display available columns
print(f"Fields in main record set: {df.columns.tolist()}")

# Attempt to auto-select a numeric field (e.g., 'MW (kDa)', 'Coverage (%)', 'Abundance [Norm]')
import numpy as np
numeric_candidates = [
    c for c in df.columns if (df[c].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[c]))
]
if not numeric_candidates:
    # Try to coerce columns with numeric-looking names
    likely_numeric = [c for c in df.columns if any(x in c.lower() for x in ["mw", "coverage", "abund", "peptide"])]
    for c in likely_numeric:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    numeric_candidates = [c for c in likely_numeric if pd.api.types.is_numeric_dtype(df[c])]

print(f"Numeric candidate fields: {numeric_candidates}")

# Choose the first available numeric field
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0]

# Define a threshold for filtering (e.g., MW (kDa) > 10)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a likely key field (e.g., 'Description' or 'Gene symbol')
possible_group_fields = [c for c in df.columns if any(key in c.lower() for key in ["description", "gene", "accession"])]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Mean {numeric_field} grouped by {group_field} (first few groups):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to a key attribute.

Below, we display a histogram of the main numeric field and, if available, a boxplot grouped by one key attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field by group_field (if available)
if 'group_field' in locals() and group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    top_groups = df[group_field].value_counts().nlargest(10).index
    sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field} (Top 10 groups)")
    plt.show()

## 6. Conclusion

- We have successfully loaded Croissant metadata and extracted the main record set from the dataset using `mlcroissant`.
- We identified fields and record sets using their `@id`, as per FAIR principles.
- We performed filtering, normalization, grouping, and basic visualization of one key numeric field.

The `mlcroissant` library enables systematic, re-usable access to rich datasets described by Croissant schemas.

Further investigation can include advanced analysis, domain-specific visualizations, or integration with biological databases.